# **JUPYTER NOTEBOOK — Image Analysis & Clustering Pipeline**

---

# **1. Setup & Imports**

```python
!pip install pillow scikit-learn hdbscan rich matplotlib numpy torch torchvision
```

```python
# Core imports
import os
from pathlib import Path
import json
import csv

# Images
from PIL import Image, ExifTags

# Math
import numpy as np

# ML & clustering
import torch
from torchvision import models, transforms
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics.pairwise import cosine_similarity
import hdbscan

# Visualization
import matplotlib.pyplot as plt

# Rich formatting (console only)
from rich.console import Console
from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeElapsedColumn
from rich.panel import Panel
from rich.table import Table

console = Console()
```

---

# **2. Load Images from Directory**

```python
def list_images(folder):
    exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tiff")
    paths = []
    for ext in exts:
        paths.extend(Path(folder).rglob(ext))
    return [p for p in paths]
```

```python
folder = input("Paste your image directory: ").strip()
paths = list_images(folder)
console.log(f"[green]Found {len(paths)} images")
```

---

# **3. Extract Image Metadata**

```python
def extract_metadata(path):
    img = Image.open(path)
    arr = np.array(img)

    data = {
        "filename": str(path),
        "format": img.format,
        "mode": img.mode,
        "width": img.size[0],
        "height": img.size[1],
        "mean": float(arr.mean()),
        "std": float(arr.std()),
        "min": int(arr.min()),
        "max": int(arr.max())
    }

    # EXIF
    exif_data = {}
    try:
        if hasattr(img, "_getexif") and img._getexif():
            for t, v in img._getexif().items():
                exif_data[ExifTags.TAGS.get(t, t)] = v
    except Exception:
        pass

    data["exif"] = exif_data
    return data
```

---

# **4. CNN Embeddings with ResNet50**

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
console.log(f"Using device: {device}")

prep = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

model = models.resnet50(weights="IMAGENET1K_V2")
model.to(device)
model.eval()

def get_embedding(path):
    img = Image.open(path).convert("RGB")
    t = prep(img).unsqueeze(0).to(device)
    with torch.no_grad():
        e = model(t).cpu().numpy().flatten()
    return e
```

---

# **5. Extract Metadata & Embeddings (Parallel or Serial)**

```python
import multiprocessing as mp

def process_single_image(path):
    meta = extract_metadata(path)
    emb = get_embedding(path)
    return meta, emb

def process_images(paths):
    metadata = []
    embeddings = []

    if device.type == "cuda":
        # Serial for CUDA-safety
        with Progress(SpinnerColumn(), TextColumn("{task.description}"), BarColumn(), TimeElapsedColumn()) as p:
            task = p.add_task("Processing images...", total=len(paths))
            for path in paths:
                m, e = process_single_image(path)
                metadata.append(m)
                embeddings.append(e)
                p.update(task, advance=1)
        return metadata, embeddings

    # CPU parallel mode
    with Progress(SpinnerColumn(), TextColumn("{task.description}"), BarColumn(), TimeElapsedColumn()) as p:
        task = p.add_task("Processing images...", total=len(paths))

        def wrapper(path):
            m, e = process_single_image(path)
            p.update(task, advance=1)
            return m, e

        with mp.Pool(mp.cpu_count()) as pool:
            results = pool.map(wrapper, paths)

    metadata = [m for m, _ in results]
    embeddings = [e for _, e in results]
    return metadata, embeddings
```

```python
metadata, embeddings = process_images(paths)
embeddings = np.array(embeddings)
console.log("[green]Metadata and embeddings extracted successfully")
```

---

# **6. PCA (Dimensionality Reduction)**

```python
pca = PCA(n_components=50)
reduced = pca.fit_transform(embeddings)
console.log("PCA complete")
```

---

# **7. KMeans Clustering**

```python
kmeans = KMeans(n_clusters=5)
kmeans_labels = kmeans.fit_predict(reduced)
console.log("[cyan]KMeans clustering done")
```

---

# **8. HDBSCAN Clustering**

```python
hdb = hdbscan.HDBSCAN(min_cluster_size=10)
hdb_labels = hdb.fit_predict(reduced)
console.log("[cyan]HDBSCAN clustering done")
```

---

# **9. Outlier Detection (Isolation Forest)**

```python
isof = IsolationForest(contamination=0.05)
outlier_pred = isof.fit_predict(embeddings)
console.log("[red]Outlier detection finished")
```

---

# **10. Similarity & Deduplication**

```python
def find_similar_images(embeddings, paths, threshold=0.90):
    sim = cosine_similarity(embeddings)
    pairs = []
    for i in range(len(paths)):
        for j in range(i+1, len(paths)):
            if sim[i, j] >= threshold:
                pairs.append((str(paths[i]), str(paths[j]), float(sim[i, j])))
    return pairs

similar = find_similar_images(embeddings, paths)
duplicates = [p for p in similar if p[2] >= 0.98]

console.log(f"Found {len(similar)} similar pairs and {len(duplicates)} duplicates")
```

---

# **11. Cluster Metrics (KMeans × HDBSCAN)**

```python
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

def cluster_metrics(labels, reduced):
    return {
        "silhouette": silhouette_score(reduced, labels),
        "davies_bouldin": davies_bouldin_score(reduced, labels),
        "calinski_harabasz": calinski_harabasz_score(reduced, labels),
    }

kmeans_metrics = cluster_metrics(kmeans_labels, reduced)

mask = hdb_labels != -1
if mask.sum() > 1:
    hdb_metrics = cluster_metrics(hdb_labels[mask], reduced[mask])
else:
    hdb_metrics = {"silhouette": np.nan, "davies_bouldin": np.nan, "calinski_harabasz": np.nan}

kmeans_metrics, hdb_metrics
```

---

# **12. Automatic Clusterizer Evaluation**

```python
winner = "KMeans" if kmeans_metrics["silhouette"] > hdb_metrics["silhouette"] else "HDBSCAN"
console.log(f"[bold yellow]Best clusterizer: {winner}")
```

---

# **13. Visualizations**

---

## **13.1 KMeans**

```python
plt.scatter(reduced[:,0], reduced[:,1], c=kmeans_labels, cmap="tab20")
plt.title("KMeans Clusters")
plt.show()
```

---

## **13.2 HDBSCAN**

```python
plt.scatter(reduced[:,0], reduced[:,1], c=hdb_labels, cmap="tab20")
plt.title("HDBSCAN Clusters")
plt.show()
```

---

## **13.3 Overlay**

```python
plt.figure(figsize=(8,6))
plt.scatter(reduced[:,0], reduced[:,1], c=kmeans_labels, cmap="tab20", alpha=0.4, label="KMeans")
plt.scatter(reduced[:,0], reduced[:,1], c=hdb_labels, cmap="tab20", marker="x", label="HDBSCAN")
plt.legend()
plt.title("Overlay: KMeans × HDBSCAN")
plt.show()
```

---

## **13.4 Heatmap KMeans × HDBSCAN**

```python
km_ids = sorted(set(kmeans_labels))
hdb_ids = sorted(set(hdb_labels))
matrix = np.zeros((len(km_ids), len(hdb_ids)))

for km, hb in zip(kmeans_labels, hdb_labels):
    matrix[km_ids.index(km), hdb_ids.index(hb)] += 1

plt.imshow(matrix, cmap="viridis")
plt.title("Cluster Heatmap: KMeans × HDBSCAN")
plt.colorbar()
plt.show()
```

---

# **14. HTML Report**

```python
def generate_html_report():
    html = f"""
    <html>
    <body>
    <h1>Image Dataset Report</h1>
    <h2>Total images: {len(paths)}</h2>
    <h2>Similar pairs: {len(similar)}</h2>
    <h2>Duplicates: {len(duplicates)}</h2>
    <h2>KMeans metrics: {kmeans_metrics}</h2>
    <h2>HDBSCAN metrics: {hdb_metrics}</h2>
    </body>
    </html>
    """
    with open("report.html", "w") as f:
        f.write(html)

generate_html_report()
console.log("[green]report.html generated")
```

---

# **15. How to Analyze Your Results**

✔ Open `clusters_kmeans.png` → grupos gerais  
✔ Open `clusters_hdbscan.png` → clusters densos + noise  
✔ Open `clusters_overlay.png` → onde os métodos concordam  
✔ Open `cluster_heatmap.png` → mapeamento entre clusterizadores  
✔ Check outliers  
✔ Check duplicates  
✔ Read the HTML summary  

---

# DONE!